# Apêndice D - Incrementando o laço de treino

Notebook do Apêndice D do livro *Build a Large Language Model (From
Scratch)*, de Sebastian Raschka, seguindo a ordem do notebook do autor
(`appendix-D/01_main-chapter-code/appendix-D.ipynb`).

O laço de treino do Capítulo 5 ganha três técnicas: **warmup** da taxa de
aprendizado, **cosine decay** e **gradient clipping**. A função completa
(`train_model`) e o cronograma (`taxa_de_aprendizado`) já estão em
`src/training/treino.py`; a inspeção dos gradientes fica em
`src/training/gradientes.py`.

Para rodar rápido em qualquer máquina, o treino deste notebook usa o
modelo **mini** do projeto. O treino com o GPT-2 124M, como no livro, fica
no script `experiments/fluxo_apendiceD.py`.

In [ ]:
import sys
from pathlib import Path

# O notebook roda a partir de Apendices/appendix-D/notebooks/: a raiz do
# projeto fica tres niveis acima.
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import torch

from src.data.tokenizer import get_bpe_tokenizer
from src.model.config import GPT_CONFIG_MINI, criar_config
from src.model.gpt_model import GPTModel
from src.training.dados import criar_loaders_treino_validacao
from src.training.dispositivo import descrever_dispositivo, escolher_dispositivo

device = escolher_dispositivo()
print("Dispositivo:", descrever_dispositivo(device))

cfg = criar_config(GPT_CONFIG_MINI)
tokenizer = get_bpe_tokenizer()
texto = (PROJECT_ROOT / "data" / "the-verdict.txt").read_text(encoding="utf-8")

torch.manual_seed(123)
train_loader, val_loader = criar_loaders_treino_validacao(texto, tokenizer, cfg, batch_size=2)
print("Lotes de treino por epoca:", len(train_loader))

## D.1 Warmup da taxa de aprendizado

No começo os pesos são aleatórios e os gradientes são grandes e pouco
confiáveis. O **warmup** começa com uma taxa pequena (`initial_lr`) e sobe
em linha reta até o pico (`peak_lr`) nos primeiros 20% dos passos.

Valores do livro: 15 épocas, `initial_lr = 0,0001` e `peak_lr = 0,01`. Com
os 9 lotes por época do GPT-2 124M (contexto 256) são 135 passos, então
`warmup_steps = int(0,2 × 135) = 27`.

In [ ]:
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

passos_por_epoca = 9                       # GPT-2 124M com contexto 256 (livro)
total_steps = passos_por_epoca * n_epochs
warmup_steps = int(0.2 * total_steps)      # 20% de warmup
print("warmup_steps:", warmup_steps)

lr_increment = (peak_lr - initial_lr) / warmup_steps
track_lrs = []
for global_step in range(total_steps):
    if global_step < warmup_steps:
        lr = initial_lr + global_step * lr_increment
    else:
        lr = peak_lr
    track_lrs.append(lr)

plt.figure(figsize=(5, 3))
plt.plot(track_lrs)
plt.ylabel("Taxa de aprendizado")
plt.xlabel("Passo")
plt.tight_layout()
plt.show()

## D.2 Cosine decay

Depois do pico, a taxa desce em meia onda de cosseno até `min_lr`: passos
grandes no meio do treino e passos finos no fim, para não estragar o que já
foi aprendido. A função `taxa_de_aprendizado` junta warmup e cosine:

```
lr = min_lr + (peak_lr − min_lr) · 0,5 · (1 + cos(π · progresso))
```

In [ ]:
from src.training.treino import taxa_de_aprendizado

min_lr = 0.1 * initial_lr
track_lrs_cos = [taxa_de_aprendizado(s, peak_lr, initial_lr, min_lr, warmup_steps, total_steps)
                 for s in range(total_steps)]

meio = warmup_steps + (total_steps - warmup_steps) // 2
print(f"passo 0     : {track_lrs_cos[0]:.2e}")
print(f"passo {warmup_steps}    : {track_lrs_cos[warmup_steps]:.2e}  (pico)")
print(f"passo {meio}    : {track_lrs_cos[meio]:.2e}  (meio do cosine)")
print(f"passo {total_steps - 1}   : {track_lrs_cos[-1]:.2e}")

plt.figure(figsize=(5, 3))
plt.plot(track_lrs, label="D.1: so warmup")
plt.plot(track_lrs_cos, label="D.2: warmup + cosine")
plt.ylabel("Taxa de aprendizado")
plt.xlabel("Passo")
plt.legend()
plt.tight_layout()
plt.show()

## D.3 Gradient clipping

Se a norma L2 de todos os gradientes juntos passar de `max_norm`, o
`clip_grad_norm_` multiplica **todos** por `max_norm / norma`. A direção do
passo continua a mesma; só o tamanho diminui. Ele devolve a norma
original, antes do corte.

In [ ]:
from src.training.gradientes import find_highest_gradient, norma_do_gradiente
from src.training.losses import calc_loss_batch

torch.manual_seed(123)
model = GPTModel(cfg).to(device)
input_batch, target_batch = next(iter(train_loader))
loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()

print(f"Maior gradiente: {find_highest_gradient(model).item():.4f}")
print(f"Norma L2       : {norma_do_gradiente(model):.4f}")

In [ ]:
# Multiplica a perda por 10 so para garantir uma norma acima de 1.0 e ver o corte
model.zero_grad()
(calc_loss_batch(input_batch, target_batch, model, device) * 10).backward()
print(f"Antes : maior {find_highest_gradient(model).item():.4f} | norma {norma_do_gradiente(model):.4f}")

devolvida = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"clip_grad_norm_ devolveu a norma original: {devolvida.item():.4f}")
print(f"Depois: maior {find_highest_gradient(model).item():.4f} | norma {norma_do_gradiente(model):.4f}")

## D.4 O laço de treino modificado

`train_model` (em `src/training/treino.py`) é o `train_model_simple` do
Capítulo 5 com as três técnicas. Como no livro, o clipping só entra depois
do warmup. Abaixo, o mesmo modelo é treinado duas vezes com o mesmo lr de
pico (0,001): uma com lr constante e outra com os extras do apêndice.

In [ ]:
from src.training.treino import train_model, train_model_simple

n_epochs = 15
peak_lr = 0.001
warmup_steps = int(0.2 * n_epochs * len(train_loader))
comum = dict(num_epochs=n_epochs, eval_freq=5, eval_iter=1,
             start_context="Every effort moves you", tokenizer=tokenizer)

torch.manual_seed(123)
model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)
hist_constante = train_model_simple(model, train_loader, val_loader, optimizer, device, **comum)

In [ ]:
torch.manual_seed(123)
model = GPTModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)
hist_extras = train_model(model, train_loader, val_loader, optimizer, device,
                          warmup_steps=warmup_steps, initial_lr=1e-5, min_lr=1e-5, **comum)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for nome, hist in (("lr constante", hist_constante), ("warmup + cosine + clip", hist_extras)):
    ax1.plot(hist["epocas"], hist["val_losses"], label=nome)
ax1.set_xlabel("Epocas"); ax1.set_ylabel("Perda de validacao"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(hist_extras["lrs"]); ax2.set_xlabel("Passo"); ax2.set_ylabel("Taxa de aprendizado"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for nome, hist in (("lr constante", hist_constante), ("warmup + cosine + clip", hist_extras)):
    print(f"{nome:<24} val final {hist['val_losses'][-1]:.3f} | melhor {min(hist['val_losses']):.3f}")